# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [ ]:
# imports

import os
import json
import uuid
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
from dateutil import parser

# For date parsing - handles various date formats beyond LLM training cutoff
try:
    from dateutil import parser
except ImportError:
    print("⚠️ Warning: python-dateutil not installed. Install with: pip install python-dateutil")
    # Fallback: simple date parsing
    parser = None

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "prices.db"

In [ ]:
#system_message = """
#You are a helpful assistant for an Airline called FlightAI.
#Give short, courteous answers, no more than 1 sentence.
#Always be accurate. If you don't know the answer, say so.
#"""


system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.

=== FLIGHT BOOKING FLOW (Two-Step Process) ===

STEP 1 - COLLECT INFORMATION:
Before generating a quote, you MUST collect ALL of the following:
1. Passenger's full name
2. Email address (for booking confirmation)
3. Origin city (where they're flying from)
4. Destination city (where they're flying to)
5. Departure date (when they want to leave)
6. Return date (for round trips - ask if one-way or round trip)
7. Preferred departure time (Morning 6AM, Mid-Morning 10AM, Afternoon 2PM, Evening 6PM, Night 9PM)
8. Class preference (Economy, Business, or First Class)
9. Number of passengers

STEP 2 - QUOTE & CONFIRMATION:
Once you have all information, call get_flight_quote to show the price breakdown.
The quote will show: base fare, class multiplier, taxes, and total price.
Ask the customer to confirm the quote before finalizing the booking.

STEP 3 - FINALIZE:
Only after customer confirms, call confirm_booking with the quote_id to finalize.
The booking confirmation will be "sent" to their email.

=== DATE HANDLING ===
- Users can book flights for ANY future date, even years ahead (2025, 2026, etc.)
- Do NOT assume dates based on your training data cutoff
- Accept dates in formats like "2025-06-15", "June 15, 2025", "next month", etc.

=== CANCELLATION ===
- Users can cancel bookings using their booking ID
- Call cancel_booking with the booking_id to process cancellation

Always clearly state the booking ID when confirming a booking.
"""

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("Paris")

In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

In [ ]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## A bit more about what Gradio actually does:

1. Gradio constructs a frontend Svelte app based on our Python description of the UI
2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this React app
3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

And of course when Gradio generates the frontend app, it ensures that the the Submit button calls the right backend route.

That's it!

It's simple, and it has a result that feels magical.

# Let's go multi-modal!!

We can use DALL-E-3, the image generation model behind GPT-4o, to make us some images

Let's put this in a function called artist.

### Price alert: each time I generate an image it costs about 4 cents - don't go crazy with images!

In [ ]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [ ]:
def artist(city):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
image = artist("New York City")
display(image)

In [ ]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

## Let's bring this home:

1. A multi-modal AI assistant with image and audio generation
2. Tool callling with database lookup
3. A step towards an Agentic workflow


In [ ]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [ ]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## The 3 types of Gradio UI

`gr.Interface` is for standard, simple UIs

`gr.ChatInterface` is for standard ChatBot UIs

`gr.Blocks` is for custom UIs where you control the components and the callbacks

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

# Exercises and Business Applications

Add in more tools - perhaps to simulate actually booking a flight. A student has done this and provided their example in the community contributions folder.

Next: take this and apply it to your business. Make a multi-modal AI assistant with tools that could carry out an activity for your work. A customer support assistant? New employee onboarding assistant? So many possibilities! Also, see the week2 end of week Exercise in the separate Notebook.

## Enhanced Realistic Flight Booking System

This implementation adds realistic features to the booking system:
- **Two-step booking flow**: Quote → Confirm (prevents accidental bookings)
- **Flight times**: Scheduled departures (6AM, 10AM, 2PM, 6PM, 9PM)
- **Class selection**: Economy, Business, First Class with price multipliers
- **Price breakdown**: Base fare, class multiplier, taxes & fees, total
- **Email confirmation**: Simulated booking confirmation to email
- **Cancellation**: Ability to cancel existing bookings

### Why this matters (Future me):
Real booking systems NEVER complete a purchase without showing the price first.
This two-step flow (quote → confirm) is industry standard and teaches
proper UX patterns for transactional AI systems.


In [ ]:
# ============================================================================
# CONFIGURATION - Flight times, classes, and pricing
# ============================================================================

# Available flight times (realistic schedule)
FLIGHT_TIMES = {
    "morning": "06:00 AM",
    "mid-morning": "10:00 AM",
    "afternoon": "02:00 PM",
    "evening": "06:00 PM",
    "night": "09:00 PM"
}

# Class multipliers (Economy = base price, Business = 2.5x, First = 4x)
CLASS_MULTIPLIERS = {
    "economy": 1.0,
    "business": 2.5,
    "first": 4.0
}

# Tax rate (realistic airline taxes ~12%)
TAX_RATE = 0.12

# Database for bookings (separate from prices.db)
BOOKINGS_DB = "bookings_v2.db"

print("✓ Configuration loaded")
print(f"  Flight times: {list(FLIGHT_TIMES.keys())}")
print(f"  Classes: {list(CLASS_MULTIPLIERS.keys())}")
print(f"  Tax rate: {TAX_RATE * 100}%")


In [ ]:
# ============================================================================
# DATABASE SCHEMA - Enhanced for realistic bookings
# ============================================================================

# Create tables for quotes and bookings
with sqlite3.connect(BOOKINGS_DB) as conn:
    cursor = conn.cursor()
    
    # Quotes table - stores pending quotes before confirmation
    # Why separate table: Real systems keep quotes for a period before they expire
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS quotes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            quote_id TEXT UNIQUE NOT NULL,
            passenger_name TEXT NOT NULL,
            email TEXT NOT NULL,
            phone TEXT,
            origin_city TEXT NOT NULL,
            destination_city TEXT NOT NULL,
            departure_date TEXT NOT NULL,
            departure_time TEXT NOT NULL,
            return_date TEXT,
            return_time TEXT,
            flight_class TEXT NOT NULL,
            num_passengers INTEGER NOT NULL,
            base_fare REAL NOT NULL,
            class_multiplier REAL NOT NULL,
            taxes REAL NOT NULL,
            total_price REAL NOT NULL,
            status TEXT DEFAULT 'pending',
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
            expires_at DATETIME
        )
    ''')
    
    # Bookings table - stores confirmed bookings
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS bookings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            booking_id TEXT UNIQUE NOT NULL,
            quote_id TEXT NOT NULL,
            passenger_name TEXT NOT NULL,
            email TEXT NOT NULL,
            phone TEXT,
            origin_city TEXT NOT NULL,
            destination_city TEXT NOT NULL,
            departure_date TEXT NOT NULL,
            departure_time TEXT NOT NULL,
            return_date TEXT,
            return_time TEXT,
            flight_class TEXT NOT NULL,
            num_passengers INTEGER NOT NULL,
            base_fare REAL NOT NULL,
            class_multiplier REAL NOT NULL,
            taxes REAL NOT NULL,
            total_price REAL NOT NULL,
            status TEXT DEFAULT 'confirmed',
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
            confirmation_sent_at DATETIME
        )
    ''')
    
    # Create indexes for fast lookups
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_quote_id ON quotes(quote_id)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_booking_id ON bookings(booking_id)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_booking_email ON bookings(email)')
    
    conn.commit()

print("✓ Database schema created (bookings_v2.db)")
print("  Tables: quotes, bookings")


In [ ]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def parse_date(date_str):
    """Parse various date formats into YYYY-MM-DD format.
    
    Why we need this: LLMs are trained on data up to a certain cutoff date.
    Users need to book flights for future dates (2025, 2026, etc.), so we must 
    parse and validate dates beyond the training cutoff.
    """
    if parser is None:
        # Fallback if dateutil not available
        try:
            datetime.strptime(date_str, "%Y-%m-%d")
            return date_str
        except:
            return date_str
    
    try:
        parsed_date = parser.parse(date_str)
        return parsed_date.strftime("%Y-%m-%d")
    except:
        return date_str

def validate_date(date_str, must_be_future=True):
    """Validate a date string and optionally check if it's in the future.
    
    Returns: (is_valid, parsed_date_or_error_message)
    """
    try:
        parsed = parse_date(date_str)
        date_obj = datetime.strptime(parsed, "%Y-%m-%d").date()
        
        if must_be_future and date_obj < datetime.now().date():
            return False, f"Date {parsed} must be in the future. Today is {datetime.now().date()}"
        
        return True, parsed
    except Exception as e:
        return False, f"Invalid date format: {date_str}. Use format like '2025-06-15' or 'June 15, 2025'"

def normalize_flight_time(time_str):
    """Convert various time descriptions to our standard time slots."""
    time_lower = time_str.lower().strip()
    
    # Direct matches
    if time_lower in FLIGHT_TIMES:
        return time_lower
    
    # Fuzzy matching
    if "6" in time_lower and ("am" in time_lower or "morning" in time_lower):
        return "morning"
    elif "10" in time_lower or "mid" in time_lower:
        return "mid-morning"
    elif "2" in time_lower or "14" in time_lower or "afternoon" in time_lower:
        return "afternoon"
    elif "6" in time_lower and ("pm" in time_lower or "evening" in time_lower):
        return "evening"
    elif "9" in time_lower or "21" in time_lower or "night" in time_lower:
        return "night"
    
    # Default to morning if unclear
    return "morning"

def normalize_class(class_str):
    """Convert various class descriptions to our standard classes."""
    class_lower = class_str.lower().strip()
    
    if "first" in class_lower:
        return "first"
    elif "business" in class_lower:
        return "business"
    else:
        return "economy"

def get_base_fare(destination_city):
    """Get base fare from prices database."""
    try:
        with sqlite3.connect(DB) as conn:
            cursor = conn.cursor()
            cursor.execute('SELECT price FROM prices WHERE city = ?', (destination_city.lower(),))
            result = cursor.fetchone()
            return result[0] if result else 500  # Default $500 if city not found
    except:
        return 500

def generate_email_confirmation(booking):
    """Generate a simulated email confirmation (what would be sent)."""
    return f"""
════════════════════════════════════════════════════════════════
                    ✈️ FlightAI BOOKING CONFIRMATION
════════════════════════════════════════════════════════════════

Dear {booking['passenger_name']},

Your flight has been successfully booked!

📋 BOOKING REFERENCE: {booking['booking_id']}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FLIGHT DETAILS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  OUTBOUND FLIGHT
  ───────────────
  Route:      {booking['origin_city']} → {booking['destination_city']}
  Date:       {booking['departure_date']}
  Time:       {booking['departure_time']}
  Class:      {booking['flight_class'].title()}
  Passengers: {booking['num_passengers']}
"""
    + (f"""
  RETURN FLIGHT
  ─────────────
  Route:      {booking['destination_city']} → {booking['origin_city']}
  Date:       {booking['return_date']}
  Time:       {booking['return_time']}
""" if booking.get('return_date') else "") + f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PRICE BREAKDOWN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Base Fare ({booking['num_passengers']} passenger(s)):  ${booking['base_fare']:.2f}
  Class ({booking['flight_class'].title()}):             x{booking['class_multiplier']}
  Subtotal:                           ${booking['base_fare'] * booking['class_multiplier']:.2f}
  Taxes & Fees (12%):                 ${booking['taxes']:.2f}
  ─────────────────────────────────────────────────────────────
  TOTAL PAID:                         ${booking['total_price']:.2f}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

This confirmation has been sent to: {booking['email']}

Thank you for choosing FlightAI! ✈️

════════════════════════════════════════════════════════════════
"""

print("✓ Helper functions loaded")


In [ ]:
# ============================================================================
# MAIN BOOKING FUNCTIONS
# ============================================================================

def get_flight_quote(
    passenger_name: str,
    email: str,
    origin_city: str,
    destination_city: str,
    departure_date: str,
    departure_time: str,
    flight_class: str,
    num_passengers: int,
    return_date: str = None,
    return_time: str = None,
    phone: str = None
) -> str:
    """
    Generate a flight quote with full price breakdown.
    
    This is STEP 1 of the booking process. The quote must be confirmed
    before a booking is finalized.
    
    Why two-step process: Real booking systems always show the price
    before charging. This prevents accidental bookings and gives users
    a chance to review before committing.
    """
    print(f"QUOTE TOOL CALLED: {origin_city} → {destination_city} for {passenger_name}", flush=True)
    
    # Validate required fields
    if not all([passenger_name, email, origin_city, destination_city, departure_date, departure_time, flight_class]):
        return "ERROR: Missing required information. Please provide: name, email, origin, destination, date, time, and class."
    
    if not num_passengers or num_passengers < 1:
        return "ERROR: Number of passengers must be at least 1."
    
    if num_passengers > 9:
        return "ERROR: For group bookings of 10+ passengers, please contact our group desk."
    
    # Validate email format (basic check)
    if "@" not in email or "." not in email:
        return f"ERROR: Invalid email format: {email}"
    
    # Validate origin != destination
    if origin_city.lower() == destination_city.lower():
        return "ERROR: Origin and destination cities cannot be the same."
    
    # Validate departure date
    valid, result = validate_date(departure_date)
    if not valid:
        return f"ERROR: {result}"
    departure_date_parsed = result
    
    # Validate return date if provided
    return_date_parsed = None
    if return_date:
        valid, result = validate_date(return_date)
        if not valid:
            return f"ERROR: {result}"
        return_date_parsed = result
        
        # Return must be after departure
        if return_date_parsed <= departure_date_parsed:
            return "ERROR: Return date must be after departure date."
    
    # Normalize time and class
    departure_time_norm = normalize_flight_time(departure_time)
    return_time_norm = normalize_flight_time(return_time) if return_time else departure_time_norm
    flight_class_norm = normalize_class(flight_class)
    
    # Calculate pricing
    base_fare_per_person = get_base_fare(destination_city)
    
    # If round trip, double the base fare
    if return_date:
        base_fare_per_person *= 2
    
    base_fare_total = base_fare_per_person * num_passengers
    class_multiplier = CLASS_MULTIPLIERS[flight_class_norm]
    subtotal = base_fare_total * class_multiplier
    taxes = subtotal * TAX_RATE
    total_price = subtotal + taxes
    
    # Generate quote ID
    quote_id = f"QT-{uuid.uuid4().hex[:8].upper()}"
    
    # Save quote to database
    try:
        with sqlite3.connect(BOOKINGS_DB) as conn:
            cursor = conn.cursor()
            cursor.execute('''
                INSERT INTO quotes (
                    quote_id, passenger_name, email, phone, origin_city, destination_city,
                    departure_date, departure_time, return_date, return_time,
                    flight_class, num_passengers, base_fare, class_multiplier, taxes, total_price
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                quote_id, passenger_name, email, phone, origin_city, destination_city,
                departure_date_parsed, FLIGHT_TIMES[departure_time_norm],
                return_date_parsed, FLIGHT_TIMES[return_time_norm] if return_date else None,
                flight_class_norm, num_passengers, base_fare_total, class_multiplier, taxes, total_price
            ))
            conn.commit()
    except Exception as e:
        return f"ERROR: Failed to create quote: {str(e)}"
    
    # Generate quote response
    trip_type = "Round Trip" if return_date else "One-Way"
    return_info = f"""
  RETURN FLIGHT
  ─────────────
  Route:      {destination_city} → {origin_city}
  Date:       {return_date_parsed}
  Time:       {FLIGHT_TIMES[return_time_norm]}
""" if return_date else ""
    
    quote = f"""
═══════════════════════════════════════════════════════════════
                    ✈️ FlightAI FLIGHT QUOTE
═══════════════════════════════════════════════════════════════

📋 QUOTE ID: {quote_id}
   (Valid for 24 hours)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PASSENGER INFORMATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Name:       {passenger_name}
  Email:      {email}
  Phone:      {phone or 'Not provided'}
  Passengers: {num_passengers}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FLIGHT DETAILS ({trip_type})
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  OUTBOUND FLIGHT
  ───────────────
  Route:      {origin_city} → {destination_city}
  Date:       {departure_date_parsed}
  Time:       {FLIGHT_TIMES[departure_time_norm]}
  Class:      {flight_class_norm.title()}
{return_info}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💰 PRICE BREAKDOWN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Base Fare ({num_passengers} pax × ${base_fare_per_person/num_passengers:.2f}):  ${base_fare_total:.2f}
  Class Upgrade ({flight_class_norm.title()}):          ×{class_multiplier}
  Subtotal:                            ${subtotal:.2f}
  Taxes & Fees ({int(TAX_RATE*100)}%):                 ${taxes:.2f}
  ─────────────────────────────────────────────────────────────
  💵 TOTAL PRICE:                      ${total_price:.2f}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

⚠️  TO CONFIRM THIS BOOKING:
    Reply with "confirm" or say "I confirm the booking"
    Reference Quote ID: {quote_id}

═══════════════════════════════════════════════════════════════
"""
    return quote

print("✓ get_flight_quote function loaded")


In [ ]:
def confirm_booking(quote_id: str) -> str:
    """
    Confirm a flight booking from a previous quote.
    
    This is STEP 2 of the booking process. Converts a quote into a confirmed booking.
    Simulates sending a confirmation email to the customer.
    
    Why separate from quote: In real systems, payment processing happens here.
    We simulate this by requiring explicit confirmation before finalizing.
    """
    print(f"CONFIRM TOOL CALLED: Confirming quote {quote_id}", flush=True)
    
    if not quote_id:
        return "ERROR: Quote ID is required to confirm a booking."
    
    # Normalize quote ID format
    quote_id = quote_id.upper()
    if not quote_id.startswith("QT-"):
        quote_id = f"QT-{quote_id}"
    
    try:
        with sqlite3.connect(BOOKINGS_DB) as conn:
            cursor = conn.cursor()
            
            # Get the quote
            cursor.execute('SELECT * FROM quotes WHERE quote_id = ?', (quote_id,))
            quote = cursor.fetchone()
            
            if not quote:
                return f"ERROR: Quote {quote_id} not found. Please request a new quote."
            
            # Extract quote data (column indices based on schema)
            quote_data = {
                'quote_id': quote[1],
                'passenger_name': quote[2],
                'email': quote[3],
                'phone': quote[4],
                'origin_city': quote[5],
                'destination_city': quote[6],
                'departure_date': quote[7],
                'departure_time': quote[8],
                'return_date': quote[9],
                'return_time': quote[10],
                'flight_class': quote[11],
                'num_passengers': quote[12],
                'base_fare': quote[13],
                'class_multiplier': quote[14],
                'taxes': quote[15],
                'total_price': quote[16],
                'status': quote[17]
            }
            
            # Check if quote is still pending
            if quote_data['status'] != 'pending':
                return f"ERROR: Quote {quote_id} has already been {quote_data['status']}. Please request a new quote."
            
            # Generate booking ID
            booking_id = f"FLT-{uuid.uuid4().hex[:8].upper()}"
            
            # Create booking
            cursor.execute('''
                INSERT INTO bookings (
                    booking_id, quote_id, passenger_name, email, phone,
                    origin_city, destination_city, departure_date, departure_time,
                    return_date, return_time, flight_class, num_passengers,
                    base_fare, class_multiplier, taxes, total_price,
                    confirmation_sent_at
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                booking_id, quote_data['quote_id'], quote_data['passenger_name'],
                quote_data['email'], quote_data['phone'], quote_data['origin_city'],
                quote_data['destination_city'], quote_data['departure_date'],
                quote_data['departure_time'], quote_data['return_date'],
                quote_data['return_time'], quote_data['flight_class'],
                quote_data['num_passengers'], quote_data['base_fare'],
                quote_data['class_multiplier'], quote_data['taxes'],
                quote_data['total_price'], datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            ))
            
            # Mark quote as confirmed
            cursor.execute('UPDATE quotes SET status = ? WHERE quote_id = ?', ('confirmed', quote_id))
            
            conn.commit()
            
            # Prepare booking data for email
            booking_data = {**quote_data, 'booking_id': booking_id}
            
            # Generate email confirmation (simulated)
            email_content = generate_email_confirmation(booking_data)
            
            return f"""
✅ BOOKING CONFIRMED!

Your booking has been finalized and a confirmation email has been sent to {quote_data['email']}.

📋 BOOKING ID: {booking_id}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📧 EMAIL CONFIRMATION (SIMULATED)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{email_content}

⚠️ Please save your Booking ID: {booking_id}
   You can use it to check your booking status or cancel if needed.
"""
    except Exception as e:
        return f"ERROR: Failed to confirm booking: {str(e)}"

print("✓ confirm_booking function loaded")


In [ ]:
def cancel_booking(booking_id: str) -> str:
    """
    Cancel an existing booking.
    
    In a real system, this would trigger refund processing based on
    cancellation policy. We simulate by updating the booking status.
    """
    print(f"CANCEL TOOL CALLED: Cancelling booking {booking_id}", flush=True)
    
    if not booking_id:
        return "ERROR: Booking ID is required to cancel a booking."
    
    # Normalize booking ID format
    booking_id = booking_id.upper()
    if not booking_id.startswith("FLT-"):
        booking_id = f"FLT-{booking_id}"
    
    try:
        with sqlite3.connect(BOOKINGS_DB) as conn:
            cursor = conn.cursor()
            
            # Get the booking
            cursor.execute('SELECT * FROM bookings WHERE booking_id = ?', (booking_id,))
            booking = cursor.fetchone()
            
            if not booking:
                return f"ERROR: Booking {booking_id} not found. Please check the booking ID."
            
            # Extract relevant data
            status = booking[18]  # status column
            passenger_name = booking[3]
            email = booking[4]
            origin_city = booking[6]
            destination_city = booking[7]
            departure_date = booking[8]
            total_price = booking[17]
            
            # Check if already cancelled
            if status == 'cancelled':
                return f"ERROR: Booking {booking_id} has already been cancelled."
            
            # Update status to cancelled
            cursor.execute('''
                UPDATE bookings SET status = ? WHERE booking_id = ?
            ''', ('cancelled', booking_id))
            
            conn.commit()
            
            return f"""
❌ BOOKING CANCELLED

Your booking has been successfully cancelled.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CANCELLATION DETAILS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Booking ID:   {booking_id}
  Passenger:    {passenger_name}
  Route:        {origin_city} → {destination_city}
  Date:         {departure_date}
  Amount:       ${total_price:.2f}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💳 REFUND INFORMATION (SIMULATED)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  A refund of ${total_price:.2f} will be processed to your
  original payment method within 5-7 business days.
  
  A cancellation confirmation email has been sent to: {email}

Thank you for choosing FlightAI. We hope to see you again!
"""
    except Exception as e:
        return f"ERROR: Failed to cancel booking: {str(e)}"


def get_booking_details(booking_id: str) -> str:
    """
    Retrieve detailed information about a booking.
    """
    print(f"DETAILS TOOL CALLED: Getting details for {booking_id}", flush=True)
    
    if not booking_id:
        return "ERROR: Booking ID is required."
    
    # Normalize booking ID format
    booking_id = booking_id.upper()
    if not booking_id.startswith("FLT-"):
        booking_id = f"FLT-{booking_id}"
    
    try:
        with sqlite3.connect(BOOKINGS_DB) as conn:
            cursor = conn.cursor()
            
            cursor.execute('SELECT * FROM bookings WHERE booking_id = ?', (booking_id,))
            booking = cursor.fetchone()
            
            if not booking:
                return f"ERROR: Booking {booking_id} not found."
            
            # Extract booking data
            b = {
                'booking_id': booking[1],
                'quote_id': booking[2],
                'passenger_name': booking[3],
                'email': booking[4],
                'phone': booking[5],
                'origin_city': booking[6],
                'destination_city': booking[7],
                'departure_date': booking[8],
                'departure_time': booking[9],
                'return_date': booking[10],
                'return_time': booking[11],
                'flight_class': booking[12],
                'num_passengers': booking[13],
                'base_fare': booking[14],
                'class_multiplier': booking[15],
                'taxes': booking[16],
                'total_price': booking[17],
                'status': booking[18],
                'created_at': booking[19]
            }
            
            trip_type = "Round Trip" if b['return_date'] else "One-Way"
            return_info = f"""
  RETURN FLIGHT
  Date:       {b['return_date']}
  Time:       {b['return_time']}
""" if b['return_date'] else ""
            
            status_emoji = "✅" if b['status'] == 'confirmed' else "❌" if b['status'] == 'cancelled' else "⏳"
            
            return f"""
═══════════════════════════════════════════════════════════════
              ✈️ FlightAI BOOKING DETAILS
═══════════════════════════════════════════════════════════════

📋 BOOKING ID: {b['booking_id']}
   Status: {status_emoji} {b['status'].upper()}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PASSENGER INFORMATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Name:       {b['passenger_name']}
  Email:      {b['email']}
  Phone:      {b['phone'] or 'Not provided'}
  Passengers: {b['num_passengers']}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FLIGHT DETAILS ({trip_type})
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OUTBOUND FLIGHT
  Route:      {b['origin_city']} → {b['destination_city']}
  Date:       {b['departure_date']}
  Time:       {b['departure_time']}
  Class:      {b['flight_class'].title()}
{return_info}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PAYMENT SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Base Fare:  ${b['base_fare']:.2f}
  Class:      ×{b['class_multiplier']}
  Taxes:      ${b['taxes']:.2f}
  ─────────────────────────────────────
  TOTAL:      ${b['total_price']:.2f}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Booked on: {b['created_at']}

═══════════════════════════════════════════════════════════════
"""
    except Exception as e:
        return f"ERROR: Failed to retrieve booking: {str(e)}"

print("✓ cancel_booking and get_booking_details functions loaded")


In [ ]:
# ============================================================================
# TOOL DEFINITIONS - OpenAI Function Calling Schema
# ============================================================================

# Tool 1: Get ticket price (simple lookup)
price_function = {
    "name": "get_ticket_price",
    "description": "Get the base price of a flight to a destination city. Use this for quick price inquiries.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to travel to"
            }
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

# Tool 2: Get flight quote (detailed pricing with all options)
quote_function = {
    "name": "get_flight_quote",
    "description": """Generate a detailed flight quote with price breakdown. 
    IMPORTANT: Only call this when you have ALL required information:
    - Passenger name and email
    - Origin and destination cities
    - Departure date and preferred time
    - Flight class preference
    - Number of passengers
    If any information is missing, ask the customer first.""",
    "parameters": {
        "type": "object",
        "properties": {
            "passenger_name": {
                "type": "string",
                "description": "Full name of the primary passenger"
            },
            "email": {
                "type": "string",
                "description": "Email address for booking confirmation"
            },
            "origin_city": {
                "type": "string",
                "description": "City where the flight departs from"
            },
            "destination_city": {
                "type": "string",
                "description": "City where the flight arrives"
            },
            "departure_date": {
                "type": "string",
                "description": "Departure date (e.g., '2025-06-15', 'June 15, 2025')"
            },
            "departure_time": {
                "type": "string",
                "description": "Preferred departure time: 'morning' (6AM), 'mid-morning' (10AM), 'afternoon' (2PM), 'evening' (6PM), or 'night' (9PM)"
            },
            "flight_class": {
                "type": "string",
                "description": "Class of travel: 'economy', 'business', or 'first'"
            },
            "num_passengers": {
                "type": "integer",
                "description": "Number of passengers (1-9)"
            },
            "return_date": {
                "type": "string",
                "description": "Return date for round trips (optional, leave empty for one-way)"
            },
            "return_time": {
                "type": "string",
                "description": "Preferred return flight time (optional)"
            },
            "phone": {
                "type": "string",
                "description": "Phone number (optional)"
            }
        },
        "required": ["passenger_name", "email", "origin_city", "destination_city", 
                    "departure_date", "departure_time", "flight_class", "num_passengers"],
        "additionalProperties": False
    }
}

# Tool 3: Confirm booking (finalize from quote)
confirm_function = {
    "name": "confirm_booking",
    "description": """Confirm and finalize a flight booking from a previous quote.
    Only call this when the customer explicitly confirms they want to proceed with the booking.
    Requires the quote_id from a previous get_flight_quote call.""",
    "parameters": {
        "type": "object",
        "properties": {
            "quote_id": {
                "type": "string",
                "description": "The quote ID to confirm (format: QT-XXXXXXXX)"
            }
        },
        "required": ["quote_id"],
        "additionalProperties": False
    }
}

# Tool 4: Cancel booking
cancel_function = {
    "name": "cancel_booking",
    "description": "Cancel an existing flight booking. Requires the booking ID.",
    "parameters": {
        "type": "object",
        "properties": {
            "booking_id": {
                "type": "string",
                "description": "The booking ID to cancel (format: FLT-XXXXXXXX)"
            }
        },
        "required": ["booking_id"],
        "additionalProperties": False
    }
}

# Tool 5: Get booking details
details_function = {
    "name": "get_booking_details",
    "description": "Retrieve details of an existing booking by booking ID.",
    "parameters": {
        "type": "object",
        "properties": {
            "booking_id": {
                "type": "string",
                "description": "The booking ID to look up (format: FLT-XXXXXXXX)"
            }
        },
        "required": ["booking_id"],
        "additionalProperties": False
    }
}

# Combine all tools
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": quote_function},
    {"type": "function", "function": confirm_function},
    {"type": "function", "function": cancel_function},
    {"type": "function", "function": details_function}
]

print("✓ Tool definitions loaded")
print(f"  Available tools: {[t['function']['name'] for t in tools]}")


In [ ]:
# ============================================================================
# TOOL HANDLER - Routes tool calls to appropriate functions
# ============================================================================

def handle_tool_calls(message):
    """
    Handle all tool calls from the LLM.
    
    This is the dispatcher that routes tool calls to the appropriate
    Python functions based on the tool name.
    """
    responses = []
    
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        
        try:
            if tool_name == "get_ticket_price":
                city = arguments.get('destination_city')
                result = get_ticket_price(city)
            
            elif tool_name == "get_flight_quote":
                result = get_flight_quote(
                    passenger_name=arguments.get('passenger_name'),
                    email=arguments.get('email'),
                    origin_city=arguments.get('origin_city'),
                    destination_city=arguments.get('destination_city'),
                    departure_date=arguments.get('departure_date'),
                    departure_time=arguments.get('departure_time'),
                    flight_class=arguments.get('flight_class'),
                    num_passengers=arguments.get('num_passengers'),
                    return_date=arguments.get('return_date'),
                    return_time=arguments.get('return_time'),
                    phone=arguments.get('phone')
                )
            
            elif tool_name == "confirm_booking":
                quote_id = arguments.get('quote_id')
                result = confirm_booking(quote_id)
            
            elif tool_name == "cancel_booking":
                booking_id = arguments.get('booking_id')
                result = cancel_booking(booking_id)
            
            elif tool_name == "get_booking_details":
                booking_id = arguments.get('booking_id')
                result = get_booking_details(booking_id)
            
            else:
                result = f"ERROR: Unknown tool '{tool_name}'"
        
        except Exception as e:
            result = f"ERROR: Tool execution failed: {str(e)}"
        
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    
    return responses

print("✓ handle_tool_calls function loaded")


In [ ]:
# ============================================================================
# CHAT FUNCTION - Main conversation loop
# ============================================================================

def chat(message, history):
    """
    Main chat function that handles the conversation with tool calling.
    
    Flow:
    1. Build message history from Gradio format
    2. Send to OpenAI with tools
    3. If tool calls requested, execute them and send results back
    4. Repeat until LLM provides final response
    """
    # Convert Gradio history format to OpenAI format
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    
    # Build messages with system prompt
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    
    # Initial API call
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )
    
    # Handle tool calls in a loop (supports multiple sequential tool calls)
    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        tool_responses = handle_tool_calls(assistant_message)
        
        # Add assistant's tool call request and tool responses to messages
        messages.append(assistant_message)
        messages.extend(tool_responses)
        
        # Continue conversation with tool results
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )
    
    return response.choices[0].message.content

print("✓ chat function loaded")


In [ ]:
# ============================================================================
# LAUNCH GRADIO INTERFACE
# ============================================================================

# Launch the chat interface with the enhanced booking system
gr.ChatInterface(
    fn=chat,
    type="messages",
    title="✈️ FlightAI - Realistic Flight Booking Assistant",
    description="""
    Welcome to FlightAI! I can help you:
    - 💰 Get flight prices
    - 📋 Generate detailed quotes with price breakdown
    - ✅ Book flights (after you confirm the quote)
    - 🔍 Check booking details
    - ❌ Cancel bookings
    
    **Booking Flow:** Tell me where you want to go → I'll ask for details → 
    Generate a quote → You confirm → Booking complete!
    """,
    examples=[
        "What's the price to Tokyo?",
        "I want to book a flight from New York to Paris",
        "Can you check my booking FLT-12345678?",
        "I need to cancel my booking"
    ]
).launch()


## Testing the Enhanced Booking System

### Complete Booking Flow Test:

**Step 1 - Start a booking:**
```
"I want to book a flight from New York to Tokyo"
```

**Step 2 - Provide details (assistant will ask for each):**
- Name: John Smith
- Email: john.smith@email.com
- Departure date: June 15, 2025
- Return date: June 25, 2025 (or say "one-way")
- Preferred time: Morning (6AM)
- Class: Economy
- Number of passengers: 2

**Step 3 - Review the quote:**
- The system will show a detailed price breakdown
- Review the total price

**Step 4 - Confirm:**
```
"Yes, I confirm the booking"
```

**Step 5 - Check your booking:**
```
"What are the details for booking FLT-XXXXXXXX?"
```

**Step 6 - (Optional) Cancel:**
```
"Cancel my booking FLT-XXXXXXXX"
```

### Features Implemented:
- ✅ Two-step booking (Quote → Confirm)
- ✅ Flight times (5 options: 6AM, 10AM, 2PM, 6PM, 9PM)
- ✅ Class selection (Economy, Business, First)
- ✅ Price breakdown (base fare, class multiplier, taxes)
- ✅ Email confirmation (simulated)
- ✅ Multiple passengers (1-9)
- ✅ Round trip and one-way
- ✅ Booking cancellation
- ✅ Booking details lookup


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a HUGE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>